In [1]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession # type: ignore

spark = SparkSession.builder \
    .appName("SparkCourse") \
    .master("local[*]") \
    .config("spark.sql.warehouse.dir", "D:/pyspark_udemy_codespace/setup/spark-warehouse") \
    .enableHiveSupport() \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print("Spark version:", spark.version)

Spark version: 3.5.0


In [2]:
"""
1. How Spark stores the timestamp
Timestamp is internally stored as 12 byte integer known as INT96
Timestamp is made up of 7 fields
Year
Month
Day
Hour
Minute
Second
Up to 6 decimal places
Microsecond precision
Timezone
"""

data_list_1 = [(1, "2022-05-18T10:30:30.0000"), (2, "2022-05-19T11:30:10.0000")] # time is in string format here    # string time is in default format
data_list_2 = [(1, "18-05-2022 10:30:30.0000"), (2, "19-05-2022 10:30:10.0000")] # time is in string format here    
data_list_3 = [(1, "2022-05-18 10:30:30.0000"), (2, "19-05-2022 10:30:10.0000")] # time is in string format here    

df_1 = (spark.createDataFrame(data_list_1).toDF("id", "string_time"))
df_2 = (spark.createDataFrame(data_list_2).toDF("id", "string_time"))
df_3 = (spark.createDataFrame(data_list_3).toDF("id", "string_time"))

df_1.show()
df_2.show()
df_3.show()

+---+--------------------+
| id|         string_time|
+---+--------------------+
|  1|2022-05-18T10:30:...|
|  2|2022-05-19T11:30:...|
+---+--------------------+

+---+--------------------+
| id|         string_time|
+---+--------------------+
|  1|18-05-2022 10:30:...|
|  2|19-05-2022 10:30:...|
+---+--------------------+

+---+--------------------+
| id|         string_time|
+---+--------------------+
|  1|2022-05-18 10:30:...|
|  2|19-05-2022 10:30:...|
+---+--------------------+



In [ ]:
"""
Convert string_time to timestamp datatype in df_1
"""

from pyspark.sql.functions import expr, col, to_timestamp # type: ignore

df_1.withColumn("converted_timestamp", to_timestamp(col("string_time"), "yyyy-MM-dd'T'HH:mm:ss.SSSS")).show()
# df_1.withColumn("converted_timestamp", expr("to_timestamp(string_time, 'yyyy-MM-ddTHH:mm:ss.SSSS')")).show()
# df_1.withColumn("converted_timestamp", expr("CAST(string_time AS timestamp)")).show()

+---+--------------------+-------------------+
| id|         string_time|converted_timestamp|
+---+--------------------+-------------------+
|  1|2022-05-18T10:30:...|2022-05-18 10:30:30|
|  2|2022-05-19T11:30:...|2022-05-19 11:30:10|
+---+--------------------+-------------------+



In [4]:
"""
Convert string_time to timestamp datatype in df_2
"""

df_2.withColumn("converted_timestamp", to_timestamp(col("string_time"), "dd-MM-yyyy HH:mm:ss.SSSS")).show()

+---+--------------------+-------------------+
| id|         string_time|converted_timestamp|
+---+--------------------+-------------------+
|  1|18-05-2022 10:30:...|2022-05-18 10:30:30|
|  2|19-05-2022 10:30:...|2022-05-19 10:30:10|
+---+--------------------+-------------------+



In [ ]:
"""
Convert string_time to timestamp datatype in df_3
Note: both records have different string_time formats
"""
from pyspark.sql.functions import try_to_timestamp, lit # type: ignore

df_3.withColumn("converted_timestamp", try_to_timestamp(col("string_time"), lit('yyyy-MM-dd HH:mm:ss.SSSS'))).show() # when this returns null, we can use NVL to convert all different formats

+---+--------------------+-------------------+
| id|         string_time|converted_timestamp|
+---+--------------------+-------------------+
|  1|2022-05-18 10:30:...|2022-05-18 10:30:30|
|  2|19-05-2022 10:30:...|               NULL|
+---+--------------------+-------------------+



In [ ]:
"""
3. Timezone information
    A timestamp without timezone information is incomplete.
    Spark offers two data types for timestamp
        TIMESTAMP
        TIMESTAMP_NTZ
    For TIMESTAMP, Spark assumes session timezone as the default when timezone is not specified
    Session timezone is specified as spark.sql.session.timeZone
"""

# Default timezone
spark.conf.get("spark.sql.session.timeZone")

'Etc/UTC'

In [10]:
"""
Changing Session timezone
"""

spark.conf.set("spark.sql.session.timeZone", "IST")

In [11]:
spark.conf.get("spark.sql.session.timeZone")

'IST'

In [12]:
spark.conf.set("spark.sql.session.timeZone", "UTC")

In [22]:
"""
Working with NTZ data
"""
event_ntz_schema = "component string, event_time string, reading string"

event_ntz_df = spark.read.format("csv")\
                        .option("header", True)\
                        .schema(event_ntz_schema)\
                        .load("/home/jovyan/work/data/machine-events-no-tz.csv")
event_ntz_df.show()

+---------+--------------------+-------+
|component|          event_time|reading|
+---------+--------------------+-------+
|   AXT594|17-05-2022 06:14:...|     23|
|   AXT594|17-05-2022 06:14:...|     25|
|   AXT594|17-05-2022 06:14:...|     21|
|   AXT594|17-05-2022 06:14:...|     22|
|   AXT594|17-05-2022 06:14:...|     25|
|   AXT594|17-05-2022 06:15:...|     23|
|   AXT594|17-05-2022 06:15:...|     24|
|   AXT594|17-05-2022 06:16:...|     24|
|   AXT594|17-05-2022 06:17:...|     21|
|   AXT594|17-05-2022 06:18:...|     22|
+---------+--------------------+-------+



In [ ]:
"""
Parse event_time into TIMESTAMP_NTZ data type
"""
from pyspark.sql.functions import to_timestamp_ntz, col, lit # type: ignore

event_valid_ntz_df = event_ntz_df.withColumn("valid_timestamp_ntz", to_timestamp_ntz(col("event_time"), lit("dd-MM-yyyy HH:mm:ss.SSSS")))
event_valid_ntz_df.show()
event_valid_ntz_df.printSchema()

+---------+--------------------+-------+--------------------+
|component|          event_time|reading| valid_timestamp_ntz|
+---------+--------------------+-------+--------------------+
|   AXT594|17-05-2022 06:14:...|     23|2022-05-17 06:14:...|
|   AXT594|17-05-2022 06:14:...|     25|2022-05-17 06:14:...|
|   AXT594|17-05-2022 06:14:...|     21|2022-05-17 06:14:...|
|   AXT594|17-05-2022 06:14:...|     22|2022-05-17 06:14:...|
|   AXT594|17-05-2022 06:14:...|     25|2022-05-17 06:14:...|
|   AXT594|17-05-2022 06:15:...|     23|2022-05-17 06:15:...|
|   AXT594|17-05-2022 06:15:...|     24|2022-05-17 06:15:...|
|   AXT594|17-05-2022 06:16:...|     24|2022-05-17 06:16:...|
|   AXT594|17-05-2022 06:17:...|     21|2022-05-17 06:17:...|
|   AXT594|17-05-2022 06:18:...|     22|2022-05-17 06:18:...|
+---------+--------------------+-------+--------------------+

root
 |-- component: string (nullable = true)
 |-- event_time: string (nullable = true)
 |-- reading: string (nullable = true)
 |--

In [ ]:
"""
valid_timestamp_ntz to valid timestamp with timezone assuming IST
"""
from pyspark.sql.functions import convert_timezone # type: ignore

stz = spark.conf.get("spark.sql.session.timeZone") # session timezone: UTC

events_df = event_valid_ntz_df.withColumn("valid_timestamp_tz", to_timestamp(convert_timezone(lit("IST"), lit(stz), col("valid_timestamp_ntz")))) # datatype for this new field is still timestamp_ntz before applying to_timestamp
events_df.show()
events_df.printSchema()

+---------+--------------------+-------+--------------------+--------------------+
|component|          event_time|reading| valid_timestamp_ntz|  valid_timestamp_tz|
+---------+--------------------+-------+--------------------+--------------------+
|   AXT594|17-05-2022 06:14:...|     23|2022-05-17 06:14:...|2022-05-17 00:44:...|
|   AXT594|17-05-2022 06:14:...|     25|2022-05-17 06:14:...|2022-05-17 00:44:...|
|   AXT594|17-05-2022 06:14:...|     21|2022-05-17 06:14:...|2022-05-17 00:44:...|
|   AXT594|17-05-2022 06:14:...|     22|2022-05-17 06:14:...|2022-05-17 00:44:...|
|   AXT594|17-05-2022 06:14:...|     25|2022-05-17 06:14:...|2022-05-17 00:44:...|
|   AXT594|17-05-2022 06:15:...|     23|2022-05-17 06:15:...|2022-05-17 00:45:...|
|   AXT594|17-05-2022 06:15:...|     24|2022-05-17 06:15:...|2022-05-17 00:45:...|
|   AXT594|17-05-2022 06:16:...|     24|2022-05-17 06:16:...|2022-05-17 00:46:...|
|   AXT594|17-05-2022 06:17:...|     21|2022-05-17 06:17:...|2022-05-17 00:47:...|
|   

In [33]:
"""
Working with tz data
Load machine-events-with-tz.csv file and show the data
"""

event_tz_schema = "component string, event_time string, reading string"

event_tz_df = spark.read.format("csv")\
                        .option("header", True)\
                        .schema(event_tz_schema)\
                        .load("/home/jovyan/work/data/machine-events-with-tz.csv")
event_tz_df.show()

+---------+--------------------+-------+
|component|          event_time|reading|
+---------+--------------------+-------+
|   AXT594|17-05-2022 06:14:...|     23|
|   AXT595|17-05-2022 06:14:...|     22|
|   AXT596|17-05-2022 06:14:...|     24|
|   AXT594|17-05-2022 06:14:...|     21|
|   AXT595|17-05-2022 06:14:...|     23|
|   AXT596|17-05-2022 06:15:...|     22|
|   AXT594|17-05-2022 06:15:...|     21|
|   AXT595|17-05-2022 06:16:...|     25|
|   AXT596|17-05-2022 06:17:...|     22|
|   AXT594|17-05-2022 06:18:...|     21|
+---------+--------------------+-------+



In [36]:
"""
Transform event_time string to timestamp
"""
event_data_df = event_tz_df.withColumn("valid_timestamp", to_timestamp(col("event_time"), "dd-MM-yyyy HH:mm:ss.SSSZ")) # Z in the end stands for timezone
event_data_df.show()
event_data_df.printSchema()

+---------+--------------------+-------+--------------------+
|component|          event_time|reading|     valid_timestamp|
+---------+--------------------+-------+--------------------+
|   AXT594|17-05-2022 06:14:...|     23|2022-05-17 06:14:...|
|   AXT595|17-05-2022 06:14:...|     22|2022-05-17 00:44:...|
|   AXT596|17-05-2022 06:14:...|     24|2022-05-17 05:14:...|
|   AXT594|17-05-2022 06:14:...|     21|2022-05-17 06:14:...|
|   AXT595|17-05-2022 06:14:...|     23|2022-05-17 00:44:...|
|   AXT596|17-05-2022 06:15:...|     22|2022-05-17 05:15:...|
|   AXT594|17-05-2022 06:15:...|     21|2022-05-17 06:15:...|
|   AXT595|17-05-2022 06:16:...|     25|2022-05-17 00:46:...|
|   AXT596|17-05-2022 06:17:...|     22|2022-05-17 05:17:...|
|   AXT594|17-05-2022 06:18:...|     21|2022-05-17 06:18:...|
+---------+--------------------+-------+--------------------+

root
 |-- component: string (nullable = true)
 |-- event_time: string (nullable = true)
 |-- reading: string (nullable = true)
 |--